In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import numpy as np
import matplotlib.pyplot as plt

# Load the MNIST dataset
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Preprocess the data
# Reshape images to include channel dimension
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Normalize pixel values to be between 0 and 1
train_images = train_images.astype('float32') / 255
test_images = test_images.astype('float32') / 255

# Convert labels to one-hot encoding
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
def build_custom_cnn():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(10, activation='softmax'))
    return model

# Build and compile the model
custom_cnn_model = build_custom_cnn()
custom_cnn_model.compile(optimizer='adam',
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [3]:
# Train the Customized CNN
history_custom_cnn = custom_cnn_model.fit(train_images, train_labels, epochs=5, batch_size=64, validation_data=(test_images, test_labels))

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8607 - loss: 0.4545 - val_accuracy: 0.9853 - val_loss: 0.0467
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9823 - loss: 0.0558 - val_accuracy: 0.9884 - val_loss: 0.0350
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9889 - loss: 0.0359 - val_accuracy: 0.9913 - val_loss: 0.0260
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9915 - loss: 0.0284 - val_accuracy: 0.9918 - val_loss: 0.0226
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9931 - loss: 0.0213 - val_accuracy: 0.9898 - val_loss: 0.0300


In [4]:
# Evaluate the Customized CNN
loss_custom_cnn, accuracy_custom_cnn = custom_cnn_model.evaluate(test_images, test_labels, verbose=0)
print(f"Customized CNN Test Accuracy: {accuracy_custom_cnn}")

# Get predictions
y_pred_custom_cnn = custom_cnn_model.predict(test_images)
y_pred_classes_custom_cnn = np.argmax(y_pred_custom_cnn, axis=1)
y_true_classes = np.argmax(test_labels, axis=1)

# Calculate performance metrics
conf_matrix_custom_cnn = confusion_matrix(y_true_classes, y_pred_classes_custom_cnn)
precision_custom_cnn = precision_score(y_true_classes, y_pred_classes_custom_cnn, average='macro')
recall_custom_cnn = recall_score(y_true_classes, y_pred_classes_custom_cnn, average='macro')
f1_custom_cnn = f1_score(y_true_classes, y_pred_classes_custom_cnn, average='macro')
# For ROC-AUC with multiclass, you typically need to calculate it per class
# or use one-vs-rest or one-vs-one approaches.
# Here's a simple example for a binary case (you'd need to adapt this for multiclass)
# roc_auc_custom_cnn = roc_auc_score(y_true_classes, y_pred_custom_cnn[:, 1]) # Example for binary

print("Customized CNN Performance Metrics:")
print(f"Confusion Matrix:\n{conf_matrix_custom_cnn}")
print(f"Precision: {precision_custom_cnn}")
print(f"Recall: {recall_custom_cnn}")
print(f"F1-Score: {f1_custom_cnn}")
# print(f"ROC-AUC: {roc_auc_custom_cnn}")

Customized CNN Test Accuracy: 0.989799976348877
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Customized CNN Performance Metrics:
Confusion Matrix:
[[ 977    0    1    0    0    0    1    1    0    0]
 [   0 1133    0    1    0    0    0    1    0    0]
 [   1    1 1016    0    0    0    0   13    1    0]
 [   0    1    1  998    0    5    0    4    1    0]
 [   0    0    0    0  970    0    0    0    0   12]
 [   1    0    0    4    0  881    1    1    0    4]
 [   5    5    0    0    2    1  944    0    1    0]
 [   0    2    1    1    0    0    0 1018    0    6]
 [   2    0    1    0    0    4    0    2  963    2]
 [   1    0    0    0    3    3    0    3    1  998]]
Precision: 0.9898716765880939
Recall: 0.9896702876338903
F1-Score: 0.9897487115679431
